# Customer 360 Accelerator
### Notebook 00 — Setup & Load Data

---

> **Purpose:** Create the Lakehouse, upload the Customer360 CSV to OneLake,
> and load it as a managed Delta table.
>
> Each step is idempotent — safe to re-run if something fails partway through.

#### Prerequisites

| # | Requirement | Detail |
|---|-------------|--------|
| 1 | **Fabric-enabled workspace** | Use this workspace for all resources |
| 2 | **Workspace role: Contributor+** | Admin recommended |
| 3 | **F2+ capacity** | Minimum for Data Agent support |
| 4 | **Tenant settings** | Enable *Ontology item (preview)*, *Data agent (preview)*, *Copilot + Azure OpenAI* |

---
## Configuration

Set `WORKSPACE_ID` and `WORKSPACE_NAME` below, or leave blank to auto-detect from the current Fabric session.

In [ ]:
# ── User Configuration ────────────────────────────────────────────────────────
WORKSPACE_ID   = ""   # Leave blank to auto-detect
WORKSPACE_NAME = ""   # Leave blank to auto-detect

LAKEHOUSE_NAME = "Customer360Lakehouse"
TABLE_NAME     = "Customer360"
CSV_FILENAME   = "customer360.csv"

In [ ]:
# ── Auto-detect workspace if not provided ────────────────────────────────────
import sempy.fabric as fabric

if not WORKSPACE_ID or not WORKSPACE_NAME:
    WORKSPACE_ID   = fabric.get_notebook_workspace_id()
    WORKSPACE_NAME = fabric.resolve_workspace_name()

print(f"Workspace ID   : {WORKSPACE_ID}")
print(f"Workspace Name : {WORKSPACE_NAME}")

---
## Step 1 — Create Lakehouse

Creates the `Customer360Lakehouse` in your workspace.  
If it already exists, the cell is a no-op — safe to re-run.

In [ ]:
client = fabric.FabricRestClient()

# List existing lakehouses
resp = client.get(f"v1/workspaces/{WORKSPACE_ID}/lakehouses")
resp.raise_for_status()
lakehouses = resp.json().get("value", [])
existing = next((lh for lh in lakehouses if lh["displayName"] == LAKEHOUSE_NAME), None)

if existing:
    LAKEHOUSE_ID = existing["id"]
    print(f"Lakehouse '{LAKEHOUSE_NAME}' already exists — skipping creation.")
    print(f"   LAKEHOUSE_ID = {LAKEHOUSE_ID}")
else:
    resp = client.post(
        f"v1/workspaces/{WORKSPACE_ID}/lakehouses",
        json={"displayName": LAKEHOUSE_NAME},
    )
    resp.raise_for_status()
    LAKEHOUSE_ID = resp.json()["id"]
    print(f"Lakehouse created: {LAKEHOUSE_NAME}")
    print(f"   LAKEHOUSE_ID = {LAKEHOUSE_ID}")

---
## Step 2 — Upload CSV to OneLake

Uploads `customer360.csv` from the notebook's local file system
to `Files/customer360.csv` in the Lakehouse via OneLake (ADLS Gen2).

In [ ]:
import os
from pathlib import Path

# The CSV should be uploaded to this notebook's file system beforehand,
# or you can use the sample-data path from the repo.
# Option A: If the CSV is in the same folder as this notebook
LOCAL_CSV_PATH = Path(f"./{CSV_FILENAME}")

# Option B: If you uploaded it to the Lakehouse Files manually, skip this step.
if not LOCAL_CSV_PATH.exists():
    # Try to find it in common locations
    for candidate in [f"./sample-data/{CSV_FILENAME}", f"/lakehouse/default/Files/{CSV_FILENAME}"]:
        if Path(candidate).exists():
            LOCAL_CSV_PATH = Path(candidate)
            break

if LOCAL_CSV_PATH.exists():
    csv_content = LOCAL_CSV_PATH.read_text(encoding="utf-8")
    ONELAKE_DEST = (
        f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com"
        f"/{LAKEHOUSE_ID}/Files/{CSV_FILENAME}"
    )
    mssparkutils.fs.put(ONELAKE_DEST, csv_content, True)
    file_size_kb = len(csv_content.encode("utf-8")) / 1024
    print(f"Uploaded {CSV_FILENAME} ({file_size_kb:.1f} KB) to Lakehouse Files/")
else:
    print(f"CSV file not found at {LOCAL_CSV_PATH}")
    print("Please upload customer360.csv to this notebook's file system or the Lakehouse Files/ folder.")

---
## Step 3 — Load CSV as Delta Table

Reads the CSV from OneLake Files/ and writes it as a managed Delta table
in the Lakehouse with proper schema inference.

In [ ]:
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, DateType
)

# Define explicit schema for Customer360
CUSTOMER360_SCHEMA = StructType([
    StructField("CustomerId",       StringType(),  False),
    StructField("FullName",         StringType(),  False),
    StructField("State",            StringType(),  True),
    StructField("City",             StringType(),  True),
    StructField("Segment",          StringType(),  True),
    StructField("LifetimeValue",    DoubleType(),  True),
    StructField("MonthlyRevenue",   DoubleType(),  True),
    StructField("ChurnRiskScore",   DoubleType(),  True),
    StructField("LastPurchaseDate", DateType(),    True),
])

# Read CSV from OneLake
csv_path = (
    f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com"
    f"/{LAKEHOUSE_ID}/Files/{CSV_FILENAME}"
)

df = (
    spark.read
    .schema(CUSTOMER360_SCHEMA)
    .option("header", "true")
    .csv(csv_path)
)

row_count = df.count()
print(f"Read {row_count} rows from {CSV_FILENAME}")

# Write as managed Delta table (overwrite for idempotency)
full_table_name = f"`{WORKSPACE_NAME}`.{LAKEHOUSE_NAME}.{TABLE_NAME}"
df.write.format("delta").mode("overwrite").saveAsTable(full_table_name)

print(f"Delta table created: {full_table_name}")
print(f"   Rows loaded: {row_count}")

---
## Verification

Quick check to confirm the table was loaded correctly.

In [ ]:
# Verify row count
verify_df = spark.table(full_table_name)
print(f"Table: {full_table_name}")
print(f"Row count: {verify_df.count()}")
print(f"Schema:")
verify_df.printSchema()

# Show sample data
print("\nSample records:")
display(verify_df.limit(5))

In [ ]:
# Store IDs for downstream notebooks
print("\n" + "=" * 60)
print("OUTPUTS — Copy these to the next notebook if needed")
print("=" * 60)
print(f"WORKSPACE_ID   = \"{WORKSPACE_ID}\"")
print(f"WORKSPACE_NAME = \"{WORKSPACE_NAME}\"")
print(f"LAKEHOUSE_ID   = \"{LAKEHOUSE_ID}\"")
print(f"LAKEHOUSE_NAME = \"{LAKEHOUSE_NAME}\"")
print(f"TABLE_NAME     = \"{TABLE_NAME}\"")